# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from haversine_build_graph_and_train import *
import torch

/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 478

In [3]:
df = pd.read_csv(f"../../../data/top30groups/OneHotLongLatCombined/combined/combined{partition}.csv")

In [4]:
from sklearn.preprocessing import StandardScaler

# Columns to exclude from scaling
exclude_cols = ['gname']

# Columns to scale
scale_cols = [col for col in df.columns if col not in exclude_cols]

# Scale only selected columns
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [5]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

# Create longlat feature

In [6]:
#geodata = ['longitude', 'latitude']
#combined_geo = df.copy()
#combined_geo['longlat'] = list(zip(df['longitude'], df['latitude']))
#combined_geo = combined_geo.drop(columns=geodata)

In [7]:
#import ast

#def to_tuple_if_needed(val):
#    if isinstance(val, str):
#        return ast.literal_eval(val)
#    return val  # already a tuple

#combined_geo['longlat'] = combined_geo['longlat'].apply(to_tuple_if_needed)

# Weapon type prediction

In [8]:
torch.cuda.empty_cache()


In [9]:
#label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}

#full_coords, data, edge_index_full, y_gcn, y_nrf, non_geo_features, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(
#        combined_geo, label_index, continuous_col='weaptype1')

In [10]:
df.columns

Index(['extended', 'vicinity', 'multiple', 'success', 'suicide', 'attacktype1',
       'targtype1', 'target1', 'individual', 'weaptype1',
       ...
       'longlat_(126.30035, 8.801867)', 'longlat_(126.314871, 8.217759)',
       'longlat_(126.549117, 7.695505)', 'longlat_(126.566013, 7.33041)',
       'longlat_(128.352878, -3.341638)', 'longlat_(135.06626, 48.502731)',
       'longlat_(140.321777, 35.772163)', 'longlat_(144.965, -37.8182)',
       'longlat_(151.179691, -33.934491)', 'gname'],
      dtype='object', length=6310)

In [11]:
label_index = {g: i for i, g in enumerate(sorted(df['gname'].unique()))}
continuous_cols = ['weaptype1']
y_preds, y_trues, logs = [], [], []
from itertools import product
import os
import random
# Hyperparameter grid
#Config: {'lr': 0.01, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}

param_grid = {
    'lr': [0.01],
    'n_tree': [10],
    'tree_depth': [14],
    'tree_feature_rate': [0.1],
    'feat_dropout': [0.1],
    'embed_dim': [32]
    }

# Convert to list of dicts (cartesian product)
grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())
sampled_combos = random.sample(grid_combos, 1)

for col in continuous_cols:
    print(f"\nTraining model for {col} prediction...")

    data,edge_index_full, y_gcn, y_nrf, train_df, val_df, test_df, train_mask, val_mask, test_mask, row_to_node_index, index_to_label = build_graph_data(df, label_index)

    best_run = None
    best_score = -1

    for combo in sampled_combos:
        args = {
            **dict(zip(param_names, combo)),
            'partition': f"gtd{partition}",
            'n_class': len(label_index),
            'epochs': 1500,
            'final_evaluation': False
        }

        print(f"Running config: {args}")
        acc, epoch, p, r, f1, y_pred_decoded, y_true_decoded, p_micro, r_micro, f1_micro, p_macro, r_macro, f1_macro, auc_w, auc_mi, auc_ma, epoch_logs = train_joint(
            df, data, edge_index_full, y_gcn, y_nrf, train_df, val_df, test_df,
                train_mask, val_mask, test_mask, args,
                row_to_node_index, index_to_label, verbose=True)

        if acc > best_score:
            best_score = acc
            best_run = {
                "args": args,
                "acc": acc,
                "epoch": epoch,
                "y_pred": y_pred_decoded,
                "y_true": y_true_decoded,
                "precision": p,
                "recall": r,
                "f1": f1,
                "micro": (p_micro, r_micro, f1_micro),
                "macro": (p_macro, r_macro, f1_macro),
                "auroc": (auc_w, auc_mi, auc_ma),
                "epoch_logs": epoch_logs
            }


    # Save best results
    if best_run:
        args = best_run["args"]
        os.makedirs(f"Results{partition}", exist_ok=True)

        results_path = f"Results{partition}/Results_{col}_prediction"
        with open(results_path, "w") as f:
            f.write(f"Best acc: {best_run['acc']:.4f} at epoch {best_run['epoch']} for {col} prediction\n")
            f.write(f"Config: {args}\n")
            f.write(f"Weighted Precision: {best_run['precision']:.4f}, Recall: {best_run['recall']:.4f}, F1: {best_run['f1']:.4f}\n")
            f.write(f"Macro Precision: {best_run['macro'][0]:.4f}, Recall: {best_run['macro'][1]:.4f}, F1: {best_run['macro'][2]:.4f}\n")
            f.write(f"Micro Precision: {best_run['micro'][0]:.4f}, Recall: {best_run['micro'][1]:.4f}, F1: {best_run['micro'][2]:.4f}\n")
            f.write(f"AUROC Weighted: {best_run['auroc'][0]:.4f}, Micro: {best_run['auroc'][1]:.4f}, Macro: {best_run['auroc'][2]:.4f}\n")

        log_path = f"Results{partition}/epoch_logs_{col}_prediction"
        with open(log_path, "w") as f:
            f.write('\n'.join(f"{x:.4f}" for x in best_run['epoch_logs']))

        y_preds.append(best_run['y_pred'])
        y_trues.append(best_run['y_true'])

print(best_score)


Training model for weaptype1 prediction...
7819


/home/jovyan/MEX0512/GTD_2025/Codes/Baselines/GraphRfi_oneHotLongLat/haversine_build_graph_and_train.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['pair'] = list(zip(df['target1'], df['weaptype1']))


tensor(4925)
tensor(1868)
tensor(1884)
tensor(11369)
Running config: {'lr': 0.01, 'n_tree': 10, 'tree_depth': 14, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}
Epoch 000 | GCN Loss: 3.4007 | NRF Loss: 3.4012 | Joint: 6.8019 | Val Acc: 0.5740
Epoch 050 | GCN Loss: 0.0444 | NRF Loss: 1.9781 | Joint: 2.0225 | Val Acc: 0.5806
Epoch 100 | GCN Loss: 0.0075 | NRF Loss: 1.0969 | Joint: 1.1044 | Val Acc: 0.5809
Epoch 150 | GCN Loss: 0.0047 | NRF Loss: 0.5861 | Joint: 0.5909 | Val Acc: 0.5917
Early stopping at epoch 169
Best validation acc: 0.5938 @ epoch 69
0.59375


In [12]:
#test 0.9355495572090149
#Running config: {'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 64, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}
#Early stopping at epoch 779
#Best validation acc: 0.9331 @ epoch 679
#{'args': {'lr': 0.001, 'n_tree': 100, 'tree_depth': 10, 'tree_feature_rate': 0.5, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd100', 'n_class': 30, 'epochs': 3000, 'final_evaluation': True}, 'acc': 0.9355495572090149,

In [13]:
print(best_run)

{'args': {'lr': 0.01, 'n_tree': 10, 'tree_depth': 14, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'embed_dim': 32, 'partition': 'gtd478', 'n_class': 30, 'epochs': 1500, 'final_evaluation': False}, 'acc': 0.59375, 'epoch': 69, 'y_pred': ["Donetsk People's Republic", 'Muslim extremists', 'Abu Sayyaf Group (ASG)', 'Corsican National Liberation Front (FLNC)', "Donetsk People's Republic", 'Islamic State of Iraq and the Levant (ISIL)', 'Taliban', 'Sikh Extremists', 'Corsican National Liberation Front (FLNC)', 'Al-Shabaab', 'Irish Republican Army (IRA)', 'Taliban', "New People's Army (NPA)", 'Taliban', 'Taliban', 'Basque Fatherland and Freedom (ETA)', 'Liberation Tigers of Tamil Eelam (LTTE)', 'Taliban', 'Manuel Rodriguez Patriotic Front (FPMR)', 'African National Congress (South Africa)', 'Taliban', 'Fulani extremists', 'Taliban', 'Abu Sayyaf Group (ASG)', 'Islamic State of Iraq and the Levant (ISIL)', 'Irish Republican Army (IRA)', 'Communist Party of India - Maoist (CPI-Maoist)', 'Shini

In [14]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1000,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': len(label_index),
    'final_evaluation': True
}
0.9287652969360352

"""

'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1000,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': len(label_index),\n    \'final_evaluation\': True\n}\n0.9287652969360352\n\n'

In [15]:
best_acc

NameError: name 'best_acc' is not defined

In [ ]:
"""
Best acc: 0.9205 at epoch 750 for weaptype1 prediction
Weighted Precision: 0.9242, Recall: 0.9205, F1: 0.9191
Macro Precision: 0.9176, Recall: 0.9107, F1: 0.9105
Micro Precision: 0.9205, Recall: 0.9205, F1: 0.9205
AUROC Weighted: 0.9967, Micro: 0.9970, Macro: 0.9963

"""

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_pred_decoded, y_true_decoded))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, continuous_col):
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    # Save the figure
    save_path = f"Results{partition}/cm_{partition}_{continuous_col}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition {partition} to {save_path}")


In [ ]:
for i in range(len(continuous_cols)):
    plot_confusion_matrix(y_preds[i], y_trues[i], sorted(df['gname'].unique()), continuous_cols[i])